## Feature exploration across SDXL Turbo, SDXL Base, and Flux

Investigate sparse activations and interventions for the supported models using the unified helpers.


### Select the model to inspect
Change `MODEL_ID` to experiment with SDXL Turbo, SDXL Base, or Flux.


In [ ]:
MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"  # @param ['stabilityai/sdxl-turbo', 'stabilityai/stable-diffusion-xl-base-1.0', 'black-forest-labs/FLUX.1-schnell']


In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import numpy as np
import torch
from PIL import Image

sys.path.insert(0, os.getcwd())

from model_interfaces import MODEL_SPECS, create_model_adapter
from utils.loaders import resolve_dtype, load_pipeline, load_sae_bundle


In [ ]:
dtype = resolve_dtype(MODEL_ID)
pipe = load_pipeline(MODEL_ID, dtype)
saes_dict, means_dict = load_sae_bundle(MODEL_ID, dtype)
adapter = create_model_adapter(pipe)
spec = adapter.spec

print(f"Using {MODEL_ID} with dtype {dtype}. Default block: {spec.default_choice}.")


In [ ]:
prompt = "A cinematic shot of a professor sloth wearing a tuxedo at a BBQ party."
num_inference_steps = min(spec.steps, 4) if spec.steps > 1 else spec.steps

image, cache = adapter.generate_with_cache(
    prompt,
    num_inference_steps,
    spec.guidance_scale,
    positions_to_cache=list(spec.code_to_block.values()),
    seed=42,
)
top_features_dict, sparse_maps_dict = adapter.process_cache(cache, saes_dict, timestep=None)
default_block_code = spec.default_choice.split()[0]
top_features = top_features_dict[default_block_code]
sparse_maps = sparse_maps_dict[default_block_code]

print(f"Top features for {default_block_code}: {top_features[:10]}")
display(image)


In [ ]:
def render_feature_heatmap(base_image: Image.Image, activation_map: np.ndarray, scale: int) -> Image.Image:
    heatmap = np.kron(activation_map, np.ones((scale, scale)))
    base_rgba = base_image.convert('RGBA')

    jet = plt.cm.jet
    cmap = jet(np.arange(jet.N))
    cmap[:1, -1] = 0
    cmap[1:, -1] = 0.6
    cmap = ListedColormap(cmap)

    norm = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)
    heatmap_rgba = cmap(norm)
    heatmap_image = Image.fromarray((heatmap_rgba * 255).astype(np.uint8))
    return Image.alpha_composite(base_rgba, heatmap_image)

plt.figure(figsize=(12, 8))
for idx, feature in enumerate(top_features[:6]):
    plt.subplot(2, 3, idx + 1)
    activation_map = sparse_maps[..., feature]
    overlay = render_feature_heatmap(image, activation_map, adapter.heatmap_scale)
    plt.imshow(overlay)
    plt.title(f"Feature {feature}")
    plt.axis('off')
plt.tight_layout()


In [ ]:
def apply_feature(prompt_text: str, feature_idx: int, strength: float):
    activation_map = sparse_maps[..., feature_idx]
    activation_tensor = torch.from_numpy(activation_map).to(dtype).to(DEVICE)

    def hook(module, input, output):
        return spec.add_feature_on_area(
            saes_dict[default_block_code],
            feature_idx,
            activation_tensor * strength,
            module,
            input,
            output,
        )

    return adapter.apply_edit(
        prompt_text,
        default_block_code,
        hook,
        num_steps=num_inference_steps,
        guidance_scale=spec.guidance_scale,
        seed=42,
    )

strengths = [-10, -5, 5, 10]
plt.figure(figsize=(12, 12))
for row, feature in enumerate(top_features[:6]):
    for col, strength in enumerate(strengths):
        plt.subplot(6, 4, row * 4 + col + 1)
        edited = apply_feature(prompt, feature, strength)
        plt.imshow(edited)
        plt.axis('off')
        if row == 0:
            plt.title(f"Strength {strength}")
        if col == 0:
            plt.ylabel(f"Feature {feature}")
plt.tight_layout()


In [ ]:
if spec.feature_icon_hook is not None:
    def feature_icon(feature_idx: int, strength: float):
        def hook(module, input, output):
            return spec.feature_icon_hook(
                saes_dict[default_block_code],
                feature_idx,
                strength * means_dict[default_block_code][feature_idx] * saes_dict[default_block_code].k,
                module,
                input,
                output,
            )

        return adapter.apply_edit(
            "",
            default_block_code,
            hook,
            num_steps=num_inference_steps,
            guidance_scale=spec.guidance_scale,
            seed=42,
        )

    plt.figure(figsize=(6, 12))
    for row, feature in enumerate(top_features[:6]):
        for col, strength in enumerate([0.5, 1.0, 1.5]):
            plt.subplot(6, 3, row * 3 + col + 1)
            icon_image = feature_icon(feature, strength)
            plt.imshow(icon_image)
            plt.axis('off')
            if row == 0:
                plt.title(f"Strength {strength}")
            if col == 0:
                plt.ylabel(f"Feature {feature}")
    plt.tight_layout()
else:
    print("Feature icon visualization is not available for this model.")
